# Baseline PII Detection: Presidio on the Combined CRAPII + TAB Corpus

**Purpose.** This notebook produces the Phase 1 baseline for the pii-kachu project. It runs Microsoft Presidio over the combined CRAPII + TAB corpus built by `python -m src.ingest.build`, scores Presidio's redactions against the gold annotations, and reports where the detector and the annotations disagree. The scoring code is model-agnostic: additional detectors (such as the planned GLiNER model) can be registered in Section 4 and are evaluated on the same documents with the same metrics, so results are directly comparable.

**Data in scope.**

| Source | Content | What is labelled | Native annotation format |
|---|---|---|---|
| `crapii` | Student essays and reflections | Selected direct PII, mostly student names, plus email, phone, ID, street address, personal URL, username | Token-level BIO, converted to character spans during ingestion |
| `tab` | European Court of Human Rights decisions | Every person, organisation, location, date, code, demographic mention, each tagged `DIRECT`, `QUASI` or `NO_MASK` | Character spans |

UNER is deliberately excluded. It labels general named entities (PER, ORG, LOC) without saying whether an entity should be redacted, so it cannot score redaction quality. Its main contribution to the project is the analytical question it raises: detecting entities is comparatively easy; deciding which entities count as PII (for example `PERSON_PRIVATE` versus `PERSON_PUBLIC`) is the harder problem. Any UNER records found in the input are dropped by the `SOURCES` setting in Section 1.

**Evaluation unit** The baseline is scored at the character-span level. Presidio outputs character spans, TAB is natively span-level, and CRAPII is converted from token-level BIO to spans upstream in `src.ingest`. To inform the open question of span-level versus token-level evaluation, the summary tables report three granularities side by side: entity spans, non-whitespace characters, and word tokens.

**Because the two sources are labelled so differently, every result is reported separately per source.**

| Section | Contents |
|---|---|
| 1. Configuration | All settings, including which models to run |
| 2. Load data | Reads the combined corpus and builds document and span tables |
| 3. Data-quality checks | Checks the gold labels for structural problems and likely missing annotations |
| 4. Detection models | Common detector interface, Presidio detector, cached prediction runs |
| 5. Evaluation | Metrics per model, split and source; threshold tuning on dev; preliminary metrics table |
| 6. Error analysis | Most frequent misses and false alarms; predictions that may indicate missing gold labels |
| 7. Cleaning report and exports | Summary of data issues with suggested actions; CSV exports |
| 8. Adding another model | Checklist for registering a new detector |

## Setup

The first cell lists install commands for reference only; it is commented out because the project virtual environment (`requirements.txt`) is expected to provide these packages. The second cell imports the libraries used throughout.

In [ ]:
# Reference only. Uncomment if a package is missing from the environment.
# !pip install presidio-analyzer pandas matplotlib tqdm
# !python -m spacy download en_core_web_lg

In [ ]:
import json, re, hashlib, random, collections, time, warnings
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Callable, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

## 1. Configuration

All run settings live in this cell; no other cell needs editing for a standard run.

- **`QUICK_TEST`**: `True` runs a small smoke test (10% of dev, stratified by source) that finishes in minutes. `False` runs the full evaluation (all of dev and test, plus 10% of train for data-cleaning purposes), which takes roughly 30 to 90 minutes on CPU for Presidio.
- **Data location**: by default the split files `train.jsonl`, `dev.jsonl` and `test.jsonl` in `data/processed/` are read. If the team works from a single combined file instead (the `combine.json` referred to in the meeting notes), `COMBINED_FILE` points to it; each record must then carry a `split` field.
- **`SOURCES`**: the sources kept for evaluation. UNER is excluded from the baseline.
- **Model switches**: `RUN_PRESIDIO` controls whether the Presidio detector is run. Each additional model gets its own `RUN_<MODEL>` switch here (see Section 8). `ANALYSIS_MODEL` selects the model used for the detailed error analysis in Section 6.
- **Scoring settings**: `BETA = 2` gives F2, which weights recall (missed PII) more heavily than precision, in line with the project goal of penalising leakage. `MATCH_THRESHOLD` is the minimum share of a span's characters that must be covered for a match.

`SEED` controls only which documents are sampled here. It is independent of the seed (42) used by `src.ingest.build` to create the splits.

In [ ]:
# ---------------- run size ----------------
QUICK_TEST = True        # True = small smoke test (minutes). False = full run (30-90 min for Presidio on CPU).

# ---------------- data ----------------
# Locates data/processed whether the notebook sits in notebooks/ or in the repository root.
DATA_DIR = next((p for p in [Path("../data/processed"), Path("./data/processed")] if p.exists()),
                Path("../data/processed"))
COMBINED_FILE = None     # e.g. DATA_DIR / "combine.json". When set, it is used instead of the split files.
SOURCES = ["crapii", "tab"]          # UNER is intentionally excluded from the baseline
SPLITS  = ["train", "dev", "test"]
print("Reading data from:", DATA_DIR.resolve())

OUTPUT_DIR = Path("./presidio_eval_outputs")   # gitignored: predictions and exports must not be committed
CACHE_DIR  = OUTPUT_DIR / "cache"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True); CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Fraction of each split sent through the detectors, stratified by source.
# Train is not needed to score Presidio; a 10% sample is included in the full run because the
# model-flagged annotation candidates (Section 6) help clean train before any model is trained on it.
SAMPLE_FRAC = ({"train": 0.0, "dev": 0.1, "test": 0.0} if QUICK_TEST
               else {"train": 0.10, "dev": 1.0, "test": 1.0})
SEED = 13

# ---------------- models ----------------
RUN_PRESIDIO   = True
# Additional models: add a RUN_<MODEL> switch here and register the model in Section 4 (see Section 8).
ANALYSIS_MODEL = "presidio"

# Presidio settings
SPACY_MODEL = "en_core_web_lg"
LANGUAGE    = "en"
PRESIDIO_SCORE_THRESHOLD = 0.35   # applied at evaluation time; Presidio itself runs at 0 and all scores are cached
ADD_CUSTOM_RECOGNIZERS   = False  # False = vanilla Presidio baseline; see Section 4.2

# ---------------- scoring ----------------
BETA            = 2.0       # F2
MATCH_THRESHOLD = 0.5       # a span "matches" if >= 50% of its non-space characters are covered
FLATTEN_GOLD    = True      # repo convention: flatten nested gold spans longest-first for entity-level metrics

# Gold spans that must be redacted. CRAPII spans have no identifier_type (None) and are all direct PII.
# TAB NO_MASK spans are excluded from the must-redact set and are scored separately as over-redaction.
MASK_IDENTIFIER_TYPES = {"DIRECT", "QUASI", None}

# Tokens that do not count as "leaked" if left unredacted (honorifics, function words)
LEAK_IGNORE_TOKENS = {"mr", "mrs", "ms", "miss", "dr", "prof", "judge", "sir", "the", "of", "and",
                      "a", "an", "in", "at", "on", "de", "la", "le", "von", "van"}

## 2. Load data

This step reads the combined CRAPII + TAB corpus in the project's unified format (one document per record: `doc_id`, `source`, `text`, `spans` with `start`, `end`, `label`, `attrs`, and `split`; TAB documents also carry `ignored_spans` for MISC and QUANTITY). Records whose source is not in `SOURCES` are dropped and counted.

Two tables are built:

- `doc_df`: one row per document.
- `gold_df`: one row per gold span (a piece of text an annotator marked).

Each gold label is also mapped to a **coarse type** (PERSON, LOCATION, ADDRESS, DATE, ORG, DEMOGRAPHIC, ID_NUM, EMAIL, PHONE, URL, OTHER). Every detector maps its own labels into the same coarse space in Section 4, which is what allows type-aware matching and model comparison. The mapping follows the unified label table agreed in the meeting:

| Unified label | CRAPII origin | TAB origin | Coarse type |
|---|---|---|---|
| `PERSON_PRIVATE` | NAME_STUDENT | PERSON + DIRECT/QUASI | PERSON |
| `PERSON_PUBLIC` | (not labelled) | PERSON + NO_MASK | PERSON |
| `ORG` | (not labelled) | ORG | ORG |
| `LOCATION_OTHER` | (not labelled) | LOC | LOCATION |
| `STREET_ADDRESS` | STREET_ADDRESS | (under LOC) | ADDRESS |
| `DATE_OTHER` | (not labelled) | DATETIME | DATE |
| `EMAIL` / `PHONE` | EMAIL / PHONE_NUM | (not labelled) | EMAIL / PHONE |
| `ID_NUM` | ID_NUM | CODE | ID_NUM |
| `URL_PERSONAL` | URL_PERSONAL | (not labelled) | URL |
| `OTHER_PII` | USERNAME | (not labelled) | OTHER |
| `DEMOGRAPHIC` | (not labelled) | DEM | DEMOGRAPHIC |

Note on `PERSON_PUBLIC`: TAB's `NO_MASK` means the span stays visible under TAB's guidelines. It does not necessarily mean the person is a public figure.

The printed table shows documents, median length, documents containing PII, and span counts per split and source.

In [ ]:
def _read_records(path):
    """Read a JSON array file or a JSONL file.

    JSONL is split on "\n" only. str.splitlines() must not be used here: it also breaks on Unicode
    separators (U+2028, U+2029, U+0085, form feed, ...), which can legitimately appear unescaped
    inside the document text and would cut a record in half.
    """
    raw = Path(path).read_text(encoding="utf-8-sig")
    if raw.lstrip().startswith("["):
        return json.loads(raw)
    records = []
    for i, line in enumerate(raw.split("\n"), start=1):
        if not line.strip():
            continue
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError as e:
            raise ValueError(f"{Path(path).name}, line {i}: invalid JSON ({e})") from e
    return records

def load_corpus():
    split_files = {s: DATA_DIR / f"{s}.jsonl" for s in SPLITS}
    records = []
    if COMBINED_FILE is not None:
        for i, d in enumerate(_read_records(COMBINED_FILE)):
            if "split" not in d:
                raise ValueError(f"record {i} in {COMBINED_FILE} has no 'split' field")
            d["_line"] = i
            records.append(d)
        origin = str(COMBINED_FILE)
    elif all(p.exists() for p in split_files.values()):
        for s, p in split_files.items():
            for i, d in enumerate(_read_records(p)):
                d["split"] = s          # trust the file name, not the field
                d["_line"] = i
                records.append(d)
        origin = ", ".join(p.name for p in split_files.values())
    else:
        raise FileNotFoundError(f"No split files in {DATA_DIR.resolve()} and COMBINED_FILE is not set. "
                                "Build them with `python -m src.ingest.build` or set COMBINED_FILE.")
    dropped = collections.Counter(d.get("source") for d in records
                                  if d.get("source") not in SOURCES or d["split"] not in SPLITS)
    kept = [d for d in records if d.get("source") in SOURCES and d["split"] in SPLITS]
    print(f"Loaded {len(records)} records from {origin}; kept {len(kept)}"
          + (f"; dropped by source/split filter: {dict(dropped)}" if dropped else ""))
    return kept

ALL_DOCS = load_corpus()
for d in ALL_DOCS:
    d["key"] = f'{d["split"]}|{d["doc_id"]}'
    d.setdefault("spans", [])
DOCS = {s: [d for d in ALL_DOCS if d["split"] == s] for s in SPLITS}
DOC_BY_KEY = {d["key"]: d for d in ALL_DOCS}

def is_must_mask(span):
    return (span.get("attrs") or {}).get("identifier_type") in MASK_IDENTIFIER_TYPES

# Unified gold label -> coarse type shared by gold and predictions
GOLD_TO_COARSE = {
    "PERSON_PRIVATE": "PERSON", "PERSON_PUBLIC": "PERSON",
    "LOCATION_OTHER": "LOCATION", "STREET_ADDRESS": "ADDRESS",
    "DATE_OTHER": "DATE", "ORG": "ORG", "DEMOGRAPHIC": "DEMOGRAPHIC",
    "ID_NUM": "ID_NUM", "EMAIL": "EMAIL", "PHONE": "PHONE",
    "URL_PERSONAL": "URL", "OTHER_PII": "OTHER",
}
# Predicted coarse types that count as the same type as a gold coarse type.
# A detector's LOCATION hit on a street address is treated as correctly typed.
GOLD_COARSE_ACCEPTS = {"ADDRESS": {"ADDRESS", "LOCATION"}}
def accepted_pred_types(gold_coarse):
    return GOLD_COARSE_ACCEPTS.get(gold_coarse, {gold_coarse})

doc_df = pd.DataFrame([{
    "key": d["key"], "doc_id": d["doc_id"], "split": d["split"], "source": d["source"],
    "n_chars": len(d["text"]), "n_spans": len(d["spans"]),
    "n_must_mask": sum(is_must_mask(s) for s in d["spans"]),
} for d in ALL_DOCS])

gold_df = pd.DataFrame([{
    "key": d["key"], "split": d["split"], "source": d["source"],
    "start": s["start"], "end": s["end"], "label": s["label"],
    "coarse": GOLD_TO_COARSE.get(s["label"], "OTHER"),
    "entity_type": (s.get("attrs") or {}).get("entity_type"),
    "identifier_type": (s.get("attrs") or {}).get("identifier_type"),
    "must_mask": is_must_mask(s),
    "text": d["text"][s["start"]:s["end"]],
} for d in ALL_DOCS for s in d["spans"]],
    columns=["key", "split", "source", "start", "end", "label", "coarse", "entity_type",
             "identifier_type", "must_mask", "text"])

unmapped = sorted(set(gold_df["label"]) - set(GOLD_TO_COARSE))
if unmapped:
    warnings.warn(f"Gold labels without a coarse mapping (treated as OTHER): {unmapped}")

display(doc_df.groupby(["split", "source"]).agg(docs=("key", "size"), median_chars=("n_chars", "median"),
        docs_with_pii=("n_must_mask", lambda x: int((x > 0).sum())), spans=("n_spans", "sum"),
        must_mask_spans=("n_must_mask", "sum")))

## 3. Data-quality checks

These checks examine the **gold labels themselves**, independently of any model. Nothing is modified; each finding is appended to `issues`, which is summarised with suggested actions in Section 7 and exported to CSV.

Structural checks performed per document:

- empty text, encoding artefacts (replacement or control characters)
- span offsets outside the text
- spans that start or end on whitespace or punctuation, or that cut through a word
- duplicate spans, and identical offsets carrying different labels
- nested and partially overlapping spans (nesting is expected in TAB)
- TAB spans missing `entity_type` or `identifier_type`, and `PERSON_PUBLIC` spans marked `DIRECT` or `QUASI`

In [ ]:
issues = []   # one row per individual finding
def flag(check, d, detail, example=""):
    issues.append({"check": check, "split": d["split"], "source": d["source"], "key": d["key"],
                   "detail": detail, "example": example[:120]})

ignored_by_key = {d["key"]: d.get("ignored_spans", []) or [] for d in ALL_DOCS}

for d in tqdm(ALL_DOCS, desc="span checks"):
    text, spans = d["text"], d["spans"]
    if not text.strip():
        flag("empty_text", d, "document text is empty")
        continue
    if "\ufffd" in text or re.search(r"[\x00-\x08\x0b\x0c\x0e-\x1f]", text):
        flag("encoding_artifacts", d, "replacement / control characters in text")

    seen = collections.Counter()
    for s in spans:
        st, en = s["start"], s["end"]
        if not (0 <= st < en <= len(text)):
            flag("bad_offsets", d, f"{st}-{en} outside text of len {len(text)}")
            continue
        surf = text[st:en]
        if surf != surf.strip():
            flag("span_whitespace_boundary", d, s["label"], repr(surf))
        if re.match(r"^[\s.,;:()\[\]\"'“”‘’-]|.*[,;:(\[\"“‘-]$", surf, flags=re.S):
            flag("span_punct_boundary", d, s["label"], repr(surf))
        if (st > 0 and text[st - 1].isalnum() and surf[:1].isalnum()) or \
           (en < len(text) and text[en].isalnum() and surf[-1:].isalnum()):
            flag("span_cuts_word", d, s["label"], text[max(0, st - 10):en + 10])
        seen[(st, en, s["label"])] += 1
    for (st, en, lab), c in seen.items():
        if c > 1:
            flag("duplicate_span", d, f"{lab} x{c}", text[st:en])

    by_off = collections.defaultdict(set)
    for s in spans:
        by_off[(s["start"], s["end"])].add(s["label"])
    for (st, en), labs in by_off.items():
        if len(labs) > 1:
            flag("same_offsets_diff_labels", d, "/".join(sorted(labs)), text[st:en])

    srt = sorted(spans, key=lambda s: (s["start"], -s["end"]))
    for a, b in zip(srt, srt[1:]):
        if b["start"] < a["end"] and (a["start"], a["end"]) != (b["start"], b["end"]):
            kind = "nested_span" if b["end"] <= a["end"] else "partial_overlap_span"
            flag(kind, d, f'{a["label"]} / {b["label"]}', f'{text[a["start"]:a["end"]]} | {text[b["start"]:b["end"]]}')

    # label <-> attribute consistency
    for s in spans:
        a = s.get("attrs") or {}
        if d["source"] == "tab" and (a.get("entity_type") is None or a.get("identifier_type") is None):
            flag("tab_missing_attrs", d, s["label"], text[s["start"]:s["end"]])
        if s["label"] == "PERSON_PUBLIC" and a.get("identifier_type") in {"DIRECT", "QUASI"}:
            flag("public_person_marked_maskable", d, a.get("identifier_type"), text[s["start"]:s["end"]])

### 3.1 Unlabelled repeat mentions

For each document, the check searches for further occurrences of strings that are already annotated as must-redact PII elsewhere in the same document but are not covered by any gold or ignored span. These are the most likely **missing annotations**, a common cleaning target, especially for student names in CRAPII.

In [ ]:
def covered_mask(n, spans):
    m = np.zeros(n, dtype=bool)
    for s in spans:
        m[max(0, s["start"]):min(n, s["end"])] = True
    return m

for d in tqdm(ALL_DOCS, desc="missed mentions"):
    text = d["text"]
    if not d["spans"]:
        continue
    cov = covered_mask(len(text), d["spans"] + ignored_by_key[d["key"]])
    surfaces = {(text[s["start"]:s["end"]], s["label"]) for s in d["spans"]
                if is_must_mask(s) and len(text[s["start"]:s["end"]].strip()) >= 3
                and s["label"] not in {"DATE_OTHER"}}
    for surf, lab in surfaces:
        pat = r"(?<!\w)" + re.escape(surf) + r"(?!\w)"
        for m in re.finditer(pat, text):
            if not cov[m.start():m.end()].any():
                flag("unlabelled_repeat_mention", d, lab,
                     text[max(0, m.start() - 30):m.end() + 30].replace("\n", " "))

### 3.2 Near-duplicate documents

The ingestion build removes exact duplicates by text hash (keeping test over dev over train). This check hashes *normalised* text (lower-cased, whitespace collapsed), so it also catches copies that differ only in spacing or line breaks. Duplicates across splits are train/test contamination, which would inflate scores for any model trained on these splits.

In [ ]:
def norm_hash(t):
    return hashlib.md5(re.sub(r"\s+", " ", t.lower()).strip().encode()).hexdigest()

doc_df["text_hash"] = [norm_hash(d["text"]) for d in ALL_DOCS]
dup_groups = doc_df.groupby("text_hash").filter(lambda g: len(g) > 1)
for h, g in dup_groups.groupby("text_hash"):
    kind = "near_duplicate_doc_cross_split" if g["split"].nunique() > 1 else "near_duplicate_doc_within_split"
    for _, r in g.iterrows():
        issues.append({"check": kind, "split": r.split, "source": r.source, "key": r.key,
                       "detail": f"group of {len(g)}: " + ", ".join(g["key"]), "example": ""})

dup_ids = doc_df[doc_df.duplicated(["split", "doc_id"], keep=False)]
for _, r in dup_ids.iterrows():
    issues.append({"check": "duplicate_doc_id", "split": r.split, "source": r.source, "key": r.key,
                   "detail": "doc_id repeated within split", "example": ""})

### 3.3 Label consistency across the corpus

This table lists surface strings (seen at least five times) that receive different labels or different masking decisions in different places. Some conflicts are legitimate (for example "Denmark" as a LOCATION versus as part of an ORG), but large groups usually indicate guideline drift between annotators.

In [ ]:
g = gold_df.assign(norm=gold_df["text"].str.lower().str.strip())
g = g[g["norm"].str.len() >= 3]
conflicts = (g.groupby(["source", "norm"])
               .agg(n=("label", "size"), labels=("label", lambda x: dict(collections.Counter(x))),
                    mask_decisions=("must_mask", lambda x: dict(collections.Counter(x))),
                    n_labels=("label", "nunique"), n_mask=("must_mask", "nunique"))
               .query("n >= 5 and (n_labels > 1 or n_mask > 1)")
               .sort_values("n", ascending=False))
print(f"{len(conflicts)} surface strings with conflicting label or masking decisions (n >= 5)")
display(conflicts.head(25))

### 3.4 Label distributions

The first table cross-tabulates TAB labels against `identifier_type` (DIRECT, QUASI, NO_MASK). The second shows each label's share of spans per split; large shifts between splits are a warning sign for the split procedure.

In [ ]:
tab_gold = gold_df.query("source == 'tab'")
if len(tab_gold):
    print("TAB label x identifier_type")
    display(pd.crosstab(tab_gold["label"], tab_gold["identifier_type"].fillna("None"), margins=True))

print("Label distribution per split (% of spans)")
display((pd.crosstab([gold_df["source"], gold_df["label"]], gold_df["split"], normalize="columns") * 100).round(2))

### 3.5 Data-quality summary

Counts of findings per check and source, followed by up to five examples for the most frequent checks.

In [ ]:
issues_df = pd.DataFrame(issues, columns=["check", "split", "source", "key", "detail", "example"])
quality_summary = (issues_df.groupby(["check", "source"]).agg(findings=("key", "size"), docs=("key", "nunique"))
                   .reset_index().sort_values("findings", ascending=False))
display(quality_summary)
for chk in quality_summary["check"].unique()[:8]:
    print(f"\n--- {chk} (examples)")
    display(issues_df[issues_df.check == chk].sample(min(5, int((issues_df.check == chk).sum())), random_state=SEED)
            [["source", "key", "detail", "example"]])

## 4. Detection models

Every detector is wrapped in the same small interface so the evaluation in Section 5 never depends on which model produced the predictions.

A `Detector` has:

| Field | Meaning |
|---|---|
| `name` | Short identifier used in tables, plots and cache filenames (for example `"presidio"`) |
| `predict` | Function `text -> list of {"start", "end", "entity_type", "score"}` with character offsets into the original text. It should return all candidates down to a low score; filtering happens at evaluation time |
| `entity_map` | Maps the model's own labels to the coarse types defined in Section 2 |
| `config` | Everything that changes the model's output (model name, label set, settings). It is hashed into the cache filename so a changed configuration never reuses stale predictions |
| `default_threshold` | Score cut-off used in the main results; Section 5.5 tunes it on dev |
| `fallback` | Optional function for labels missing from `entity_map` |

`register(detector)` adds it to `DETECTORS`. Section 4.3 then runs every registered detector over the same sampled documents.

In [ ]:
COARSE_TYPES = ["PERSON", "LOCATION", "ADDRESS", "DATE", "ORG", "DEMOGRAPHIC",
                "ID_NUM", "EMAIL", "PHONE", "URL", "OTHER"]

@dataclass
class Detector:
    name: str
    predict: Callable[[str], list]
    entity_map: dict
    config: Any
    default_threshold: float = 0.5
    fallback: Optional[Callable[[str], str]] = None

    def to_coarse(self, entity_type):
        if entity_type in self.entity_map:
            return self.entity_map[entity_type]
        return self.fallback(entity_type) if self.fallback else "OTHER"

    @property
    def cache_path(self):
        tag = hashlib.md5(json.dumps(self.config, sort_keys=True, default=str).encode()).hexdigest()[:8]
        return CACHE_DIR / f"{self.name}_preds_{tag}.jsonl"

DETECTORS = {}
def register(detector):
    bad = {v for v in detector.entity_map.values() if v not in COARSE_TYPES}
    if bad:
        raise ValueError(f"{detector.name}: entity_map uses unknown coarse types {bad}")
    DETECTORS[detector.name] = detector
    print(f"registered detector '{detector.name}' (default threshold {detector.default_threshold}, "
          f"cache {detector.cache_path.name})")

### 4.1 Presidio configuration

Presidio is configured with spaCy `en_core_web_lg` and an explicit spaCy-to-Presidio entity mapping, so the run is reproducible across Presidio versions.

By default Presidio multiplies spaCy `ORG` scores by 0.4, which places them around 0.34, below most thresholds. Here that down-weighting is switched off (`low_score_entity_names` is empty) and the cut-off is controlled at evaluation time through `PRESIDIO_SCORE_THRESHOLD` instead. The printed list of supported entities should include `ORGANIZATION`.

`ADD_CUSTOM_RECOGNIZERS = False` keeps the run a vanilla Presidio baseline. Setting it to `True` adds two example pattern recognizers (ECHR application numbers and social-media handles) that illustrate cheap rule-based gains; any such change produces a separate cache file and must be reported as a separate configuration, not as the baseline.

In [ ]:
if RUN_PRESIDIO:
    from presidio_analyzer import AnalyzerEngine, PatternRecognizer, Pattern
    from presidio_analyzer.nlp_engine import NlpEngineProvider

    nlp_config = {
        "nlp_engine_name": "spacy",
        "models": [{"lang_code": LANGUAGE, "model_name": SPACY_MODEL}],
        "ner_model_configuration": {
            "model_to_presidio_entity_mapping": {
                "PER": "PERSON", "PERSON": "PERSON", "NORP": "NRP",
                "FAC": "LOCATION", "LOC": "LOCATION", "GPE": "LOCATION", "LOCATION": "LOCATION",
                "ORG": "ORGANIZATION", "ORGANIZATION": "ORGANIZATION",
                "DATE": "DATE_TIME", "TIME": "DATE_TIME",
            },
            "low_confidence_score_multiplier": 0.4,
            "low_score_entity_names": [],
            "labels_to_ignore": ["CARDINAL", "EVENT", "LANGUAGE", "LAW", "MONEY",
                                 "ORDINAL", "PERCENT", "PRODUCT", "QUANTITY", "WORK_OF_ART"],
        },
    }
    nlp_engine = NlpEngineProvider(nlp_configuration=nlp_config).create_engine()
    analyzer = AnalyzerEngine(nlp_engine=nlp_engine, supported_languages=[LANGUAGE])

    if ADD_CUSTOM_RECOGNIZERS:
        # ECHR application numbers ("no. 36244/06"); vanilla Presidio has no recognizer for these
        analyzer.registry.add_recognizer(PatternRecognizer(
            supported_entity="ECHR_APP_NO",
            patterns=[Pattern("echr_app_no", r"\b\d{1,6}/\d{2}\b", 0.6)],
            context=["application", "applications", "no", "nos"]))
        # Social-media handles such as "@zzadkth"
        analyzer.registry.add_recognizer(PatternRecognizer(
            supported_entity="SOCIAL_HANDLE", patterns=[Pattern("handle", r"(?<!\w)@[A-Za-z0-9_]{3,30}\b", 0.5)]))

    print("Presidio supported entities:", sorted(analyzer.get_supported_entities()))

### 4.2 Presidio detector

Presidio entity types are mapped to the coarse types. Entities without an explicit mapping fall back to `ID_NUM` when their name suggests an identifier (SSN, passport, credit card, IBAN and similar) and to `OTHER` otherwise. The printed dictionary shows the final mapping for every entity Presidio supports in the installed version.

The cache-tag composition is kept identical to the previous version of this notebook, so existing Presidio caches remain valid.

In [ ]:
if RUN_PRESIDIO:
    PRESIDIO_TO_COARSE = {
        "PERSON": "PERSON", "LOCATION": "LOCATION", "DATE_TIME": "DATE", "NRP": "DEMOGRAPHIC",
        "ORGANIZATION": "ORG", "EMAIL_ADDRESS": "EMAIL", "PHONE_NUMBER": "PHONE", "URL": "URL",
        "ECHR_APP_NO": "ID_NUM", "SOCIAL_HANDLE": "OTHER",   # handles correspond to CRAPII USERNAME -> OTHER_PII
    }
    _ID_HINTS = ("SSN", "NUMBER", "LICENSE", "PASSPORT", "CARD", "IBAN", "CRYPTO", "NHS", "ITIN",
                 "IP_ADDRESS", "CODE", "_ID", "AADHAAR", "PAN", "NRIC", "ABN", "ACN", "TFN", "MEDICARE")
    def presidio_fallback(entity_type):
        return "ID_NUM" if any(h in entity_type for h in _ID_HINTS) else "OTHER"

    def presidio_predict(text):
        res = analyzer.analyze(text=text, language=LANGUAGE, score_threshold=0.0)
        return [{"start": r.start, "end": r.end, "entity_type": r.entity_type, "score": round(float(r.score), 4)}
                for r in res]

    presidio = Detector(
        name="presidio",
        predict=presidio_predict,
        entity_map=PRESIDIO_TO_COARSE,
        config=["presidio", SPACY_MODEL, nlp_config, ADD_CUSTOM_RECOGNIZERS],
        default_threshold=PRESIDIO_SCORE_THRESHOLD,
        fallback=presidio_fallback,
    )
    register(presidio)
    print({e: presidio.to_coarse(e) for e in sorted(analyzer.get_supported_entities())})

### 4.3 Run all registered detectors (cached)

Documents are sampled once per split, stratified by source, according to `SAMPLE_FRAC`. Every registered detector runs on exactly the same documents, which keeps the comparison fair.

Predictions are appended to a per-detector cache file in `presidio_eval_outputs/cache/` as they are produced. An interrupted run resumes where it stopped, and raising `SAMPLE_FRAC` later only processes the new documents. Documents that raise an error are recorded in `FAILURES` and skipped rather than stopping the run. Only documents with predictions from **every** registered detector are evaluated.

The printed table shows the number of documents evaluated per split and source.

In [ ]:
def sample_docs():
    rng = random.Random(SEED)
    out = []
    for split in SPLITS:
        frac = SAMPLE_FRAC.get(split, 0)
        if frac <= 0:
            continue
        by_src = collections.defaultdict(list)
        for d in DOCS[split]:
            by_src[d["source"]].append(d)
        for src, lst in sorted(by_src.items()):
            k = len(lst) if frac >= 1 else max(1, round(len(lst) * frac))
            out.extend(rng.sample(lst, k) if k < len(lst) else lst)
    return out

def load_cache(path):
    cache = {}
    if path.exists():
        with path.open(encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    r = json.loads(line); cache[r["key"]] = r["preds"]
    return cache

def run_detector(detector, docs):
    path = detector.cache_path
    cache = load_cache(path)
    todo = [d for d in docs if d["key"] not in cache]
    print(f"[{detector.name}] {len(docs)} docs requested, {len(docs) - len(todo)} cached, {len(todo)} to run -> {path.name}")
    failures = []
    t0 = time.time()
    with path.open("a", encoding="utf-8") as f:
        for d in tqdm(todo, desc=detector.name):
            try:
                preds = detector.predict(d["text"])
            except Exception as e:                       # keep going; record the failure
                failures.append((d["key"], repr(e))); continue
            cache[d["key"]] = preds
            f.write(json.dumps({"key": d["key"], "preds": preds}) + "\n")
    if todo:
        print(f"[{detector.name}] ran {len(todo)} docs in {time.time() - t0:.0f}s; failures: {len(failures)}")
    return {d["key"]: cache[d["key"]] for d in docs if d["key"] in cache}, failures

if not DETECTORS:
    raise RuntimeError("No detectors registered. Set RUN_PRESIDIO (or another RUN_<MODEL> switch) to True.")

SAMPLED_DOCS = sample_docs()
PREDS, FAILURES = {}, {}
for name, det in DETECTORS.items():
    PREDS[name], FAILURES[name] = run_detector(det, SAMPLED_DOCS)

EVAL_DOCS = [d for d in SAMPLED_DOCS if all(d["key"] in PREDS[m] for m in DETECTORS)]
MODELS = list(DETECTORS)
if ANALYSIS_MODEL not in DETECTORS:
    ANALYSIS_MODEL = MODELS[0]
print(f"models: {MODELS}; detailed error analysis uses: {ANALYSIS_MODEL}")
display(pd.Series(collections.Counter((d["split"], d["source"]) for d in EVAL_DOCS), name="docs evaluated").sort_index())

## 5. Evaluation

Predictions are compared with the gold annotations for every model, split and source. The summary table has one row per model, view, split and source.

**Headline columns**

| Column | Question answered | Better is |
|---|---|---|
| `span_F2` | Overall entity-level score, weighted towards recall (catching PII) | higher |
| `span_recall` | Share of gold PII spans that were detected | higher |
| `span_precision` | Share of predicted spans that fall on real PII | higher |
| `leak_rate_span` | Share of gold PII spans with at least one meaningful word left visible (the privacy metric) | lower |
| `over_redaction_rate` | Share of non-PII characters that were redacted | lower |
| `nomask_redaction_rate` | TAB only: share of `NO_MASK` spans that were redacted anyway | lower |

**Two views of the predictions.** `raw` keeps every prediction. `scoped` drops predictions of coarse types that a source never annotates (for example, dates in CRAPII), because the gold data cannot say whether those are right or wrong. The `scoped` view is the fair comparison and is used for the headline numbers; `raw` shows how much extra text the detector would redact in practice.

**Three granularities** are reported to inform the span-level versus token-level question raised in the meeting:

- **Span (entity) level** (`span_*`): the primary metric for the baseline.
- **Character level** (`char_*`): non-whitespace characters only.
- **Token level** (`token_*`): word tokens (`\w+`); a token counts as PII if any of its characters is gold PII and as redacted if any of its characters is predicted.

**Scoring rules**

- Characters inside TAB `ignored_spans` (MISC, QUANTITY) are excluded from character and token metrics: redacting them is neither rewarded nor penalised.
- Nested gold spans are flattened longest-first for entity-level metrics (`FLATTEN_GOLD`), so a name nested inside an ORG is not counted twice. Character and token metrics use the union of all spans, so nesting does not matter there.
- A gold span is **detected** if at least `MATCH_THRESHOLD` of its characters are covered by any prediction, and **detected type-aware** if that coverage comes from predictions of a compatible coarse type.
- A prediction is a **true positive** if at least `MATCH_THRESHOLD` of its non-ignored characters fall on must-redact gold.
- A gold span **leaks** if any meaningful word token (honorifics and function words excluded) is not fully redacted. This is stricter than "detected": `Mr Henrik [REDACTED]` counts as detected but leaked.
- **Over-redaction** is the share of non-PII characters that were redacted. For TAB, redaction of `NO_MASK` spans is also reported, and separately for `PERSON_PUBLIC` (TAB persons marked `NO_MASK`; not necessarily public figures). This is the over-redaction case the Phase 2 LLM verifier is meant to catch.

In [ ]:
SCOPE = {src: set().union(*(accepted_pred_types(c) for c in gold_df.loc[gold_df.source == src, "coarse"].unique()))
         for src in gold_df.source.unique()}
print("scoped view keeps predicted types:", {k: sorted(v) for k, v in SCOPE.items()})

_TOK = re.compile(r"\w+", flags=re.U)

def filter_preds(preds, detector, source, threshold, view):
    out = []
    for p in preds:
        if p["score"] < threshold:
            continue
        c = detector.to_coarse(p["entity_type"])
        if view == "scoped" and c not in SCOPE.get(source, set()):
            continue
        out.append({**p, "coarse": c})
    return out

def flatten_longest_first(spans):
    """Repo convention for nested TAB spans: keep the longest span, drop any span overlapping a kept one."""
    kept = []
    for s in sorted(spans, key=lambda s: (-(s["end"] - s["start"]), s["start"])):
        if all(s["end"] <= k["start"] or s["start"] >= k["end"] for k in kept):
            kept.append(s)
    return sorted(kept, key=lambda s: s["start"])

def any_in_ranges(mask, starts, ends):
    """For each [start, end) range, True if any element of mask is True."""
    cs = np.concatenate(([0], np.cumsum(mask, dtype=np.int64)))
    return (cs[ends] - cs[starts]) > 0

_GOLD_CACHE = {}
def gold_view(d):
    """Prediction-independent arrays for a document, computed once and reused across models and thresholds."""
    k = d["key"]
    if k not in _GOLD_CACHE:
        text, n = d["text"], len(d["text"])
        nonspace = np.fromiter((not ch.isspace() for ch in text), dtype=bool, count=n)
        must = [s for s in d["spans"] if is_must_mask(s)]
        nomask = [s for s in d["spans"] if not is_must_mask(s)]
        must_m = covered_mask(n, must) & nonspace
        ign_m = covered_mask(n, ignored_by_key[k]) & nonspace & ~must_m
        toks = list(_TOK.finditer(text))
        t_st = np.array([m.start() for m in toks], dtype=np.int64)
        t_en = np.array([m.end() for m in toks], dtype=np.int64)
        tok_gold = any_in_ranges(must_m, t_st, t_en)
        tok_ign = any_in_ranges(ign_m, t_st, t_en) & ~tok_gold
        _GOLD_CACHE[k] = dict(
            nonspace=nonspace, must=must, nomask=nomask, must_m=must_m, ign_m=ign_m,
            nomask_m=covered_mask(n, nomask), clean_m=nonspace & ~must_m & ~ign_m,
            gold_eval=flatten_longest_first(must) if FLATTEN_GOLD else must,
            t_st=t_st, t_en=t_en, tok_gold=tok_gold, tok_ign=tok_ign)
    return _GOLD_CACHE[k]

def evaluate_doc(d, preds):
    text, n = d["text"], len(d["text"])
    G = gold_view(d)
    nonspace, must_m, ign_m = G["nonspace"], G["must_m"], G["ign_m"]
    pred_m = covered_mask(n, preds) & nonspace
    typed = collections.defaultdict(lambda: np.zeros(n, dtype=bool))
    for p in preds:
        typed[p["coarse"]][p["start"]:p["end"]] = True

    tok_pred = any_in_ranges(pred_m, G["t_st"], G["t_en"])
    tok_gold, tok_ign = G["tok_gold"], G["tok_ign"]
    doc = {"key": d["key"], "split": d["split"], "source": d["source"],
           "char_tp": int((pred_m & must_m).sum()), "char_fp": int((pred_m & G["clean_m"]).sum()),
           "char_fn": int((must_m & ~pred_m).sum()), "clean_chars": int(G["clean_m"].sum()),
           "tok_tp": int((tok_pred & tok_gold).sum()), "tok_fp": int((tok_pred & ~tok_gold & ~tok_ign).sum()),
           "tok_fn": int((tok_gold & ~tok_pred).sum()),
           "n_must": len(G["must"]), "n_pred": len(preds)}

    gold_rows = []
    for s in G["gold_eval"]:
        st, en = s["start"], s["end"]
        ns = nonspace[st:en]
        denom = max(1, int(ns.sum()))
        cov = (pred_m[st:en] & ns).sum() / denom
        gold_coarse = GOLD_TO_COARSE.get(s["label"], "OTHER")
        typed_m = np.zeros(en - st, dtype=bool)
        for c in accepted_pred_types(gold_coarse):
            if c in typed:
                typed_m |= typed[c][st:en]
        cov_t = (typed_m & ns).sum() / denom
        toks = [m for m in _TOK.finditer(text, st, en) if m.group().lower() not in LEAK_IGNORE_TOKENS] \
               or list(_TOK.finditer(text, st, en))
        leaked = [m.group() for m in toks if not pred_m[m.start():m.end()].all()]
        if not toks:
            leaked = [text[st:en]] if cov < 1 else []
        gold_rows.append({"key": d["key"], "split": d["split"], "source": d["source"], "label": s["label"],
                          "identifier_type": (s.get("attrs") or {}).get("identifier_type"),
                          "text": text[st:en], "coverage": cov, "detected": cov >= MATCH_THRESHOLD,
                          "detected_typed": cov_t >= MATCH_THRESHOLD, "n_tokens": max(1, len(toks)),
                          "n_leaked_tokens": len(leaked), "leaked": bool(leaked)})

    pred_rows = []
    for p in preds:
        st, en = p["start"], p["end"]
        scorable = nonspace[st:en] & ~ign_m[st:en]
        if not scorable.any():
            continue                                    # prediction sits entirely in ignored spans
        on_pii = (must_m[st:en] & scorable).sum() / scorable.sum()
        pred_rows.append({"key": d["key"], "split": d["split"], "source": d["source"],
                          "start": st, "end": en, "entity_type": p["entity_type"], "coarse": p["coarse"],
                          "score": p["score"], "text": text[st:en], "tp": on_pii >= MATCH_THRESHOLD,
                          "on_nomask": bool(G["nomask_m"][st:en].any())})

    nomask_rows = [{"key": d["key"], "split": d["split"], "source": d["source"], "label": s["label"],
                    "redacted": (pred_m[s["start"]:s["end"]] & nonspace[s["start"]:s["end"]]).sum()
                                / max(1, nonspace[s["start"]:s["end"]].sum()) >= MATCH_THRESHOLD}
                   for s in G["nomask"]]
    doc["any_leak"] = any(g["leaked"] for g in gold_rows)
    doc["any_direct_leak"] = any(g["leaked"] and g["identifier_type"] in ("DIRECT", None) for g in gold_rows)
    return doc, gold_rows, pred_rows, nomask_rows

EMPTY_COLS = {
    "gold": ["key", "split", "source", "label", "identifier_type", "text", "coverage", "detected",
             "detected_typed", "n_tokens", "n_leaked_tokens", "leaked"],
    "pred": ["key", "split", "source", "start", "end", "entity_type", "coarse", "score", "text", "tp", "on_nomask"],
    "nomask": ["key", "split", "source", "label", "redacted"],
}

def evaluate(docs, model, threshold=None, views=("raw", "scoped")):
    """Score one model's cached predictions on `docs`. Returns DataFrames keyed docs / gold / pred / nomask."""
    det = DETECTORS[model]
    threshold = det.default_threshold if threshold is None else threshold
    out = {k: [] for k in ("docs", "gold", "pred", "nomask")}
    for view in views:
        for d in docs:
            preds = filter_preds(PREDS[model][d["key"]], det, d["source"], threshold, view)
            doc, g, p, nm = evaluate_doc(d, preds)
            tag = {"model": model, "view": view, "threshold": threshold}
            for name, rows in (("docs", [doc]), ("gold", g), ("pred", p), ("nomask", nm)):
                out[name].extend({**r, **tag} for r in rows)
    frames = {}
    for k, v in out.items():
        cols = EMPTY_COLS.get(k, [])
        frames[k] = pd.DataFrame(v) if v else pd.DataFrame(columns=cols + ["model", "view", "threshold"])
    return frames

def evaluate_all(docs, models=None, threshold=None, views=("raw", "scoped")):
    parts = [evaluate(docs, m, threshold, views) for m in (models or MODELS)]
    return {k: pd.concat([p[k] for p in parts], ignore_index=True) for k in parts[0]}

def fbeta(p, r, beta=BETA):
    if r is not None and not np.isnan(r) and r == 0:
        return 0.0                      # nothing found (including "no predictions at all") scores zero
    if p is None or r is None or np.isnan(p) or np.isnan(r):
        return np.nan
    return 0.0 if p + r == 0 else (1 + beta**2) * p * r / (beta**2 * p + r)

def _ratio(a, b):
    return a / b if b else np.nan

def summarise(R, by=("model", "view", "split", "source")):
    by = list(by)
    rows = []
    def select(df, sel):
        if not len(df):
            return df
        return df.loc[np.logical_and.reduce([df[k] == v for k, v in sel.items()])]
    for keys, dg in R["docs"].groupby(by):
        sel = dict(zip(by, keys if isinstance(keys, tuple) else (keys,)))
        g, p, nm = select(R["gold"], sel), select(R["pred"], sel), select(R["nomask"], sel)
        span_r = g["detected"].mean() if len(g) else np.nan
        span_p = p["tp"].mean() if len(p) else np.nan
        ctp, cfp, cfn = dg.char_tp.sum(), dg.char_fp.sum(), dg.char_fn.sum()
        ttp, tfp, tfn = dg.tok_tp.sum(), dg.tok_fp.sum(), dg.tok_fn.sum()
        char_p, char_r = _ratio(ctp, ctp + cfp), _ratio(ctp, ctp + cfn)
        tok_p, tok_r = _ratio(ttp, ttp + tfp), _ratio(ttp, ttp + tfn)
        has = dg[dg.n_must > 0]; clean = dg[dg.n_must == 0]
        direct = g[(g.identifier_type == "DIRECT") | g.identifier_type.isna()] if len(g) else g
        rows.append({**sel, "threshold": dg["threshold"].iloc[0],
            "docs": len(dg), "gold_spans": len(g), "pred_spans": len(p),
            "span_precision": span_p, "span_recall": span_r,
            "span_recall_typed": g["detected_typed"].mean() if len(g) else np.nan,
            "span_F2": fbeta(span_p, span_r),
            "char_precision": char_p, "char_recall": char_r, "char_F2": fbeta(char_p, char_r),
            "token_precision": tok_p, "token_recall": tok_r, "token_F2": fbeta(tok_p, tok_r),
            "over_redaction_rate": _ratio(cfp, dg.clean_chars.sum()),
            "nomask_redaction_rate": nm["redacted"].mean() if len(nm) else np.nan,
            "public_person_redaction_rate": nm.loc[nm.label == "PERSON_PUBLIC", "redacted"].mean()
                                            if len(nm) and (nm.label == "PERSON_PUBLIC").any() else np.nan,
            "false_alarm_doc_rate": (clean.n_pred > 0).mean() if len(clean) else np.nan,
            "leak_rate_span": g["leaked"].mean() if len(g) else np.nan,
            "leak_rate_token": _ratio(g.n_leaked_tokens.sum(), g.n_tokens.sum()) if len(g) else np.nan,
            "leak_rate_direct": direct["leaked"].mean() if len(direct) else np.nan,
            "leak_rate_doc": has["any_leak"].mean() if len(has) else np.nan,
            "direct_leak_rate_doc": has["any_direct_leak"].mean() if len(has) else np.nan})
    return pd.DataFrame(rows)

COUNT_COLS = {"model", "view", "split", "source", "threshold", "setting", "docs", "gold_spans", "pred_spans"}
def show_metrics(df):
    pct = [c for c in df.columns if c not in COUNT_COLS and pd.api.types.is_numeric_dtype(df[c])]
    display(df.style.format({**{c: "{:.1%}" for c in pct}, "threshold": "{:.2f}"}, na_rep="-"))

### 5.1 Summary at each model's default threshold

One row per model, view (`raw` or `scoped`), split and source, at each detector's `default_threshold`. The `scoped` rows are the fair comparison.

In [ ]:
R = evaluate_all(EVAL_DOCS)
summary = summarise(R)
show_metrics(summary)

### 5.2 Recall and leakage by PII type (scoped view)

The same metrics broken down by gold label, showing which kinds of PII each model handles well or poorly. ID numbers in TAB are expected to score near zero for vanilla Presidio, because it has no recognizer for court application numbers. `recall_type_aware` counts a detection only when the predicted coarse type is compatible with the gold type; a large gap between `recall` and `recall_type_aware` means the model finds the text but labels it as the wrong kind of entity.

In [ ]:
def recall_by_type(R, view="scoped", extra=()):
    g = R["gold"][R["gold"].view == view]
    keys = ["model", "source", "label", *extra]
    return (g.groupby(keys, dropna=False).agg(n=("detected", "size"), recall=("detected", "mean"),
                             recall_type_aware=("detected_typed", "mean"), leak_rate=("leaked", "mean"),
                             mean_coverage=("coverage", "mean"))
              .sort_values(["model", "source", "n"], ascending=[True, True, False]))

RBT_FMT = {"recall": "{:.1%}", "recall_type_aware": "{:.1%}", "leak_rate": "{:.1%}", "mean_coverage": "{:.1%}"}
rbt = recall_by_type(R)
display(rbt.style.format(RBT_FMT))

rbt_tab = recall_by_type(R, extra=("identifier_type",))
if "tab" in rbt_tab.index.get_level_values("source"):
    print("TAB, by label x identifier_type")
    display(rbt_tab.xs("tab", level="source").style.format(RBT_FMT))

The chart below plots recall (any type) and leak rate per gold label, one panel per model and source.

In [ ]:
panels = [(m, s) for m in MODELS for s in SOURCES
          if (m, s) in set(zip(rbt.index.get_level_values("model"), rbt.index.get_level_values("source")))]
if panels:
    ncols = min(2, len(panels)); nrows = int(np.ceil(len(panels) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(7 * ncols, 4.5 * nrows), squeeze=False)
    for ax, (m, src) in zip(axes.flat, panels):
        t = rbt.loc[(m, src)]
        x = np.arange(len(t)); w = 0.38
        ax.bar(x - w/2, t["recall"], w, label="recall (any type)")
        ax.bar(x + w/2, t["leak_rate"], w, label="leak rate")
        ax.set_xticks(x); ax.set_xticklabels([f"{l}\n(n={n})" for l, n in zip(t.index, t.n)], rotation=45, ha="right", fontsize=8)
        ax.set_ylim(0, 1); ax.set_title(f"{m} | {src} | scoped view"); ax.legend(fontsize=8); ax.grid(axis="y", alpha=.3)
    for ax in list(axes.flat)[len(panels):]:
        ax.axis("off")
    plt.tight_layout(); plt.show()

### 5.3 Over-redaction of spans that should stay visible (TAB `NO_MASK`)

TAB marks some spans as `NO_MASK`: under its guidelines they remain visible (for example many judges, courts and country names). This table shows how often each model redacted them anyway. This is the over-redaction case that the Phase 2 LLM verifier is meant to send back for correction.

In [ ]:
nm = R["nomask"][R["nomask"].view == "scoped"]
if len(nm):
    display(nm.groupby(["model", "source", "label"]).agg(n=("redacted", "size"), redaction_rate=("redacted", "mean"))
              .sort_values(["model", "n"], ascending=[True, False]).style.format({"redaction_rate": "{:.1%}"}))
else:
    print("No NO_MASK spans in the evaluated documents.")

### 5.4 Precision by predicted entity type (raw view)

For each model, which predicted entity types produce false positives. `hits_nomask` is the share of predictions that touch a `NO_MASK` span; `median_score` helps judge whether a higher threshold would remove the errors.

In [ ]:
p_raw = R["pred"][R["pred"].view == "raw"]
prec = (p_raw.groupby(["model", "source", "entity_type"]).agg(n=("tp", "size"), precision=("tp", "mean"),
         hits_nomask=("on_nomask", "mean"), median_score=("score", "median"))
         .sort_values(["model", "source", "n"], ascending=[True, True, False]))
display(prec.style.format({"precision": "{:.1%}", "hits_nomask": "{:.1%}", "median_score": "{:.2f}"}))

### 5.5 Score-threshold sweep on dev, then test at the tuned threshold

Each prediction carries a confidence score. The sweep evaluates cut-offs from 0.00 to 0.95 on **dev only** (scoped view), per model and source, and selects the threshold with the highest span F2 (ties are broken by the lower span leak rate, then by the higher threshold, which redacts less for the same score). Lower thresholds reduce leakage at the cost of more over-redaction; the plots show that trade-off. The selected thresholds are then applied to **test**. Thresholds are never tuned on test.

In `QUICK_TEST` mode no test documents are sampled, so the test table is skipped.

In [ ]:
tune_docs = [d for d in EVAL_DOCS if d["split"] == "dev"]
if not tune_docs:
    warnings.warn("No dev documents evaluated; tuning on all evaluated documents instead.")
    tune_docs = EVAL_DOCS

THRESHOLDS = np.round(np.arange(0.0, 1.0, 0.05), 2)
sweep = []
for m in MODELS:
    for th in tqdm(THRESHOLDS, desc=f"threshold sweep ({m})"):
        s = summarise(evaluate(tune_docs, m, threshold=th, views=("scoped",)), by=("model", "source"))
        sweep.append(s)
sweep = pd.concat(sweep, ignore_index=True)

# Best threshold per model and source: highest span F2; ties broken by lower leak rate, then by the
# higher threshold (less redaction for the same score).
valid = sweep.dropna(subset=["span_F2"])
best_th = (valid.sort_values(["span_F2", "leak_rate_span", "threshold"], ascending=[False, True, False])
                .groupby(["model", "source"]).head(1)
                .sort_values(["model", "source"])
                [["model", "source", "threshold", "span_F2", "char_F2", "token_F2", "leak_rate_span", "over_redaction_rate"]]
                .reset_index(drop=True))

groups = list(sweep.groupby(["model", "source"]))
ncols = min(2, len(groups)); nrows = int(np.ceil(len(groups) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(7 * ncols, 4 * nrows), squeeze=False)
for ax, ((m, src), t) in zip(axes.flat, groups):
    for col in ["span_precision", "span_recall", "span_F2", "char_F2", "token_F2", "leak_rate_span", "over_redaction_rate"]:
        ax.plot(t.threshold, t[col], marker=".", label=col)
    b = best_th[(best_th.model == m) & (best_th.source == src)]
    if len(b):
        b = b.iloc[0]
        ax.axvline(b.threshold, ls="--", c="grey")
        ax.set_title(f"{m} | {src}: best span F2 {b.span_F2:.3f} @ {b.threshold:.2f}")
    ax.set_xlabel("score threshold"); ax.set_ylim(0, 1); ax.grid(alpha=.3); ax.legend(fontsize=7)
for ax in list(axes.flat)[len(groups):]:
    ax.axis("off")
plt.tight_layout(); plt.show()

display(best_th)

In [ ]:
test_docs = [d for d in EVAL_DOCS if d["split"] == "test"]
tuned = pd.DataFrame()
if test_docs:
    parts = []
    for _, b in best_th.iterrows():
        docs_src = [d for d in test_docs if d["source"] == b.source]
        if docs_src:
            parts.append(summarise(evaluate(docs_src, b.model, threshold=b.threshold, views=("scoped",))))
    if parts:
        tuned = pd.concat(parts, ignore_index=True)
        print("Test results at the dev-tuned threshold (scoped view)")
        show_metrics(tuned)
else:
    print("No test documents in this run (QUICK_TEST mode); test results at the tuned threshold are skipped.")

### 5.6 Preliminary metrics for the presentation

A compact table of the headline metrics, one row per model, source and setting, saved as `prelim_metrics.csv`. It contains:

- `default threshold`: each split evaluated at the detector's default threshold (scoped view).
- `dev-tuned threshold (test)`: test evaluated at the threshold selected on dev in 5.5 (full run only).

The selection of columns is a proposal; the final set of preliminary metrics is to be confirmed by the group before the slides are produced. Adding or removing a column in `PRELIM_COLS` is the only change needed.

In [ ]:
PRELIM_COLS = ["model", "setting", "split", "source", "threshold", "docs", "gold_spans",
               "span_precision", "span_recall", "span_F2", "char_F2", "token_F2",
               "leak_rate_span", "leak_rate_direct", "over_redaction_rate", "nomask_redaction_rate"]

prelim_parts = [summary[summary.view == "scoped"].assign(setting="default threshold")]
if len(tuned):
    prelim_parts.append(tuned.assign(setting="dev-tuned threshold (test)"))
prelim = (pd.concat(prelim_parts, ignore_index=True)
            .reindex(columns=PRELIM_COLS)
            .sort_values(["source", "model", "setting", "split"])
            .reset_index(drop=True))
show_metrics(prelim)
prelim.to_csv(OUTPUT_DIR / "prelim_metrics.csv", index=False)

## 6. Error analysis and cleaning candidates

This section examines the errors of `ANALYSIS_MODEL` (Presidio by default) in the scoped view:

- **Missed PII** (false negatives): the gold strings most frequently leaked.
- **False alarms** (false positives): the predicted strings most frequently not matching gold.

Model errors and annotation errors are mixed together in these lists. Two signals suggest an annotation gap rather than a model error:

1. A high-confidence prediction that looks exactly like PII, where gold says it is not.
2. A false-positive string that is labelled as PII elsewhere in the same document.

Predictions matching either signal are collected as **annotation review candidates**. Strings that are false positives in many documents (for example "Mind Mapping" or "Human Rights") are systematic model errors and are excluded from the candidates.

Two CRAPII caveats:

- CRAPII labels only **student** names. Names of instructors, authors and quoted speakers are intentionally unlabelled and, in this release, replaced by surrogates, so they look like real private names. Flagged CRAPII `PERSON` hits are therefore decisions between "over-redaction" and "missing label" to be made against `configs/labels.yaml`, not automatic fixes.
- CRAPII's `URL_PERSONAL` covers only *personal* URLs, so hits on course links (Coursera, YouTube) are usually correct false positives. They are placed at the bottom of the review list.

In [ ]:
print(f"Error analysis for model: {ANALYSIS_MODEL}")
g_err = R["gold"][(R["gold"].view == "scoped") & (R["gold"].model == ANALYSIS_MODEL)]
print("Most frequently leaked gold strings (false negatives)")
display(g_err[g_err.leaked].groupby(["source", "label", "text"]).size().rename("times_leaked")
        .sort_values(ascending=False).groupby(level=[0, 1]).head(8).to_frame())

p_err = R["pred"][(R["pred"].view == "scoped") & (R["pred"].model == ANALYSIS_MODEL)]
print("Most frequent false-positive strings")
display(p_err[~p_err.tp].groupby(["source", "entity_type", "text"]).size().rename("times")
        .sort_values(ascending=False).groupby(level=[0, 1]).head(8).to_frame())

In [ ]:
gold_surfaces = gold_df[gold_df.must_mask].groupby("key")["text"].apply(lambda s: set(s.str.lower()))
fp = p_err[~p_err.tp.astype(bool)].copy()
fp["labelled_elsewhere_in_doc"] = [t.lower() in gold_surfaces.get(k, set()) for k, t in zip(fp.key, fp.text)]
name_like = re.compile(r"^[A-Z][a-z'’-]+(?: [A-Z][a-z'’-]+){1,3}$")
fp["name_like"] = fp.text.map(lambda t: bool(name_like.match(t.strip())))

fp_doc_freq = fp.groupby(fp.text.str.lower())["key"].nunique()
fp["fp_doc_freq"] = fp.text.str.lower().map(fp_doc_freq)
MAX_FP_DOC_FREQ = 3

annotation_candidates = fp[
    (fp.fp_doc_freq <= MAX_FP_DOC_FREQ) & (
        fp.labelled_elsewhere_in_doc
        | ((fp.source == "crapii") & (fp.coarse == "PERSON") & fp.name_like & (fp.score >= 0.85))
        | ((fp.source == "crapii") & fp.coarse.isin(["EMAIL", "PHONE", "URL", "ID_NUM"]) & (fp.score >= 0.5)))
].copy()
# Review order: string already labelled in the doc > contact details and IDs > names > URLs
annotation_candidates["priority"] = np.select(
    [annotation_candidates.labelled_elsewhere_in_doc, annotation_candidates.coarse.isin(["EMAIL", "PHONE", "ID_NUM"]),
     annotation_candidates.coarse == "PERSON"], [0, 1, 2], 3)
annotation_candidates = annotation_candidates.sort_values(["priority", "score"], ascending=[True, False])
print(f"{len(annotation_candidates)} predictions that may indicate MISSING gold annotations")
display(annotation_candidates[["source", "key", "entity_type", "score", "text", "labelled_elsewhere_in_doc", "fp_doc_freq"]].head(30))

The ten highest-priority candidates are printed below with surrounding context; the predicted span is enclosed in `[[ ]]`.

In [ ]:
def context(key, start, end, width=60):
    t = DOC_BY_KEY[key]["text"]
    return (t[max(0, start - width):start] + "[[" + t[start:end] + "]]" + t[end:end + width]).replace("\n", " ")

annotation_candidates = annotation_candidates.assign(
    context=[context(k, a, b) for k, a, b in zip(annotation_candidates.key, annotation_candidates.start, annotation_candidates.end)])
for _, r in annotation_candidates.head(10).iterrows():
    print(f"[{r.source} {r.key} {r.entity_type} {r.score:.2f}] ...{r.context}...")

## 7. Cleaning report and exports

The cleaning report answers whether the data needs cleaning before Phase 2. It combines the data-quality findings from Section 3 with the model-flagged annotation candidates from Section 6 into one table: each type of problem, how many findings and documents it affects, and a suggested action.

In [ ]:
for _, r in annotation_candidates.iterrows():
    issues.append({"check": "possible_missing_annotation (model-flagged)", "split": r.split, "source": r.source,
                   "key": r.key, "detail": f"{ANALYSIS_MODEL} {r.entity_type} score={r.score:.2f}",
                   "example": r.context[:120]})
issues_df = pd.DataFrame(issues, columns=["check", "split", "source", "key", "detail", "example"])

SUGGESTED_ACTION = {
    "bad_offsets": "Fix or drop; spans cannot be scored.",
    "empty_text": "Drop document.",
    "encoding_artifacts": "Re-decode source text; re-check offsets afterwards.",
    "span_whitespace_boundary": "Strip span and shift offsets.",
    "span_punct_boundary": "Trim punctuation, or accept; only inflates exact-match false negatives.",
    "span_cuts_word": "Inspect; usually a tokenisation bug in the BIO-to-span conversion.",
    "duplicate_span": "Deduplicate.",
    "same_offsets_diff_labels": "Pick one label per guideline.",
    "nested_span": "Expected in TAB (repo flattens longest-first); only a problem when converting to BIO training data.",
    "partial_overlap_span": "Resolve manually; longest-first flattening keeps only one of the pair.",
    "tab_missing_attrs": "Fill identifier_type or exclude from masking metrics.",
    "public_person_marked_maskable": "Check against TAB guidelines.",
    "unlabelled_repeat_mention": "Likely missing annotation; add span (propagate within document).",
    "near_duplicate_doc_cross_split": "Normalise whitespace/case before hashing in src.ingest.build dedup (keeps test > dev > train).",
    "near_duplicate_doc_within_split": "Same fix in src.ingest.build dedup.",
    "duplicate_doc_id": "Make ids unique.",
    "possible_missing_annotation (model-flagged)": "Manual review; add span if PII.",
}
cleaning_report = (issues_df.groupby(["check", "source"]).agg(findings=("key", "size"), docs_affected=("key", "nunique"))
                   .reset_index())
cleaning_report["suggested_action"] = cleaning_report["check"].map(SUGGESTED_ACTION)
cleaning_report = cleaning_report.sort_values("findings", ascending=False)
display(cleaning_report)

### 7.1 Exports

All tables are written to `presidio_eval_outputs/` as CSV. This folder is gitignored; predictions and data-derived outputs must not be committed.

| File | Contents |
|---|---|
| `prelim_metrics.csv` | Headline metrics for the presentation (Section 5.6) |
| `metrics_summary.csv` | Full summary, all models, views, splits and sources (5.1) |
| `test_metrics_tuned.csv` | Test results at the dev-tuned thresholds (5.5, full run only) |
| `recall_by_type.csv` | Recall and leakage per gold label (5.2) |
| `precision_by_entity.csv` | Precision per predicted entity type (5.4) |
| `threshold_sweep_dev.csv` | Threshold sweep on dev (5.5) |
| `best_thresholds_dev.csv` | Selected threshold per model and source (5.5) |
| `gold_span_results.csv`, `pred_span_results.csv` | Per-span results for further analysis |
| `cleaning_report.csv`, `data_issues_detail.csv` | Data-quality summary and every individual finding (Section 7) |
| `annotation_review_candidates.csv` | Model-flagged possible missing annotations with context (Section 6) |

In [ ]:
summary.to_csv(OUTPUT_DIR / "metrics_summary.csv", index=False)
if len(tuned):
    tuned.to_csv(OUTPUT_DIR / "test_metrics_tuned.csv", index=False)
rbt.to_csv(OUTPUT_DIR / "recall_by_type.csv")
prec.to_csv(OUTPUT_DIR / "precision_by_entity.csv")
sweep.to_csv(OUTPUT_DIR / "threshold_sweep_dev.csv", index=False)
best_th.to_csv(OUTPUT_DIR / "best_thresholds_dev.csv", index=False)
issues_df.to_csv(OUTPUT_DIR / "data_issues_detail.csv", index=False)
cleaning_report.to_csv(OUTPUT_DIR / "cleaning_report.csv", index=False)
annotation_candidates.to_csv(OUTPUT_DIR / "annotation_review_candidates.csv", index=False)
R["gold"].to_csv(OUTPUT_DIR / "gold_span_results.csv", index=False)
R["pred"].to_csv(OUTPUT_DIR / "pred_span_results.csv", index=False)
print("wrote:", *sorted(f.name for f in OUTPUT_DIR.glob("*.csv")), sep="\n  ")

## 8. Adding another model

Sections 2, 3 and 5 to 7 are model-agnostic. Adding a detector requires changes only in Section 1 and Section 4:

1. **Write a predict function** `text -> [{"start", "end", "entity_type", "score"}]`. Offsets must be character offsets into the original, unmodified document text. Models with a limited input length (most transformer models; TAB judgments run to several thousand words) need to split long documents into overlapping chunks and shift the returned offsets back to the full document. Running the model at a low score cut-off and returning every candidate keeps the threshold sweep in Section 5.5 meaningful. Models without scores can return `1.0` for every span; the threshold sweep then becomes flat.
2. **Map its labels** to the coarse types in `COARSE_TYPES` through `entity_map` (and optionally `fallback`). `register` rejects unknown coarse types.
3. **List everything that changes its output in `config`** (checkpoint name, label prompts, thresholds used at inference, chunk sizes). The cache filename is derived from it.
4. **Register it** with `register(Detector(...))` in a new cell placed after 4.2 and before 4.3, guarded by a `RUN_<MODEL>` switch added to Section 1. The Presidio cells in 4.1 and 4.2 serve as the reference implementation.
5. **Re-run from Section 4.3.** The new model is sampled on the same documents, cached, and appears automatically in every table, plot and CSV, with a `model` column distinguishing it. `ANALYSIS_MODEL` selects which model's errors Section 6 inspects.

Changing the scoring rules (Section 5) affects every model equally. Any such change should be agreed on so that numbers remain comparable with previously reported results.